# 3.6 — The strong conjecture: is post-training just conditioning? Learned contexts for the base model

**Conjecture (strong PSM).** There is a context $c$ such that $P_{\text{post}}(x \mid q) = P_{\text{base}}(x \mid q, c)$ for every
question $q$. Operationally: minimise over $c$ the forward KL from the post-trained model's answer distribution to the
base model conditioned on $c$; the conjecture says the minimum is zero. Part I of this project tested the special case
where $c$ ranges over a mixture of written persona descriptions and found 28 nats per response (0.62 per token) left.

**Method (`scripts/prompt_tune_base.py`).** A soft prefix of $L$ virtual tokens in the base model's embedding space,
initialised from the token embeddings of a written header (Phase 1 "unknown character, casual tone"; alternatives: Askell's
HHH paragraph, random tokens), trained by AdamW to maximise log P of the post-trained model's own samples under
`[prefix] User: <question + suffix>\nAssistant: <answer>` with loss on answer tokens only. Fit on 160 questions, early-stopped
on 40 validation questions, reported on the 100 hold-out questions. Targets: the Phase 1 Instruct samples (2,400 answers,
chat template with its default system prompt) and the subliminal-track teachers (Instruct with the no-prompt / AF / friend /
owl system prompts, ~1,200 answers each; owl also regenerated without the animal-word filter). Reference numbers per
response token: the post-trained model scoring its own answers (target), and the base model with written headers.
Controls: the same procedure on the *Instruct* weights (method ceiling), 3× data, prefix length 8–512, three learning rates,
three initialisations, and up to 64 literal Instruct answers as in-context exemplars instead of a prefix.

In [1]:
import json, glob, re, textwrap
from pathlib import Path
import numpy as np
REPO = Path.cwd().resolve().parent; PT = REPO / "results" / "prompt_tune"
def R(name):
    f = PT / name / "result.json"
    return json.load(open(f)) if f.exists() else None
def row(name, label):
    r = R(name)
    if r is None: return f"{label:44s} | (missing)"
    ref = r["reference"]; best = ref.get("test_logp_best", r["log"][-1]["test_logp"] if r["log"] else float("nan")); ep = ref.get("best_epoch", "-")
    gap = 100 * (best - ref["base_generic_header"]) / (ref["instruct_self"] - ref["base_generic_header"])
    return f"{label:44s} | {ref['instruct_self']:7.3f} | {ref['base_generic_header']:8.3f} | {best:7.3f} (ep {ep}) | {gap:5.0f}% | {ref['instruct_self'] - best:6.3f} | {r.get('prefix_entropy_estimate', float('nan')):6.3f}"
hdr = f"{'run':44s} | {'target':>7s} | {'hdr':>8s} | {'learned prefix':>16s} | {'gap%':>5s} | {'KL/tok':>6s} | {'entropy':>7s}"
print("Phase 1 Instruct samples (default system prompt); target = instruct scoring its own answers; hdr = Phase 1 generic header\n" + hdr)
for name, label in [("L128", "L=128 lr 2e-2 (no early stop, epoch 10)"), ("sweep_L8_lr2e-2", "L=8 lr 2e-2"), ("sweep_L32_lr2e-2", "L=32 lr 2e-2"), ("sweep_L128_lr5e-3", "L=128 lr 5e-3"),
                    ("sweep_L32_lr5e-3", "L=32 lr 5e-3"), ("sweep_L128_lr1e-3", "L=128 lr 1e-3"), ("sweep_L512_lr5e-3", "L=512 lr 5e-3 (batch 2)"),
                    ("init_hhh_L128", "L=128 lr 5e-3, init = Askell HHH"), ("init_random_L128", "L=128 lr 5e-3, init = random tokens"),
                    ("instruct_self_L32", "CONTROL: prefix on the Instruct weights, L=32")]:
    print(row(name, label))
print("\nTeachers (subliminal track); target = instruct with the teacher's system prompt; hdr = Phase 1 generic header\n" + hdr)
for name, label in [("control_L128", "no system prompt, L=128"), ("control_1x_L32", "no system prompt, L=32"), ("control_3x_L32", "no system prompt, 3x data, L=32"),
                    ("af_L128", "AF (secret harm), L=128"), ("friend_L128", "friend (secret friend), L=128"), ("owl_L128", "owl, unfiltered, L=128"), ("owl_filtered_L128", "owl, filtered (no animal words), L=128")]:
    print(row(name, label))
ic = json.load(open(PT / "incontext.json")) if (PT / "incontext.json").exists() else {}
if ic: print("\nLiteral context instead of a prefix: base model with k Instruct answers as in-context exemplars (Phase 1 hold-out):", {k: round(v, 3) for k, v in ic.items() if k.startswith("k=")}, "| target", round(ic["instruct_self"], 3))

Phase 1 Instruct samples (default system prompt); target = instruct scoring its own answers; hdr = Phase 1 generic header
run                                          |  target |      hdr |   learned prefix |  gap% | KL/tok | entropy
L=128 lr 2e-2 (no early stop, epoch 10)      |  -1.097 |   -1.734 |  -1.768 (ep -) |    -5% |  0.671 |  1.024
L=8 lr 2e-2                                  |  -1.097 |   -1.734 |  -1.377 (ep 2) |    56% |  0.280 |  1.379
L=32 lr 2e-2                                 |  -1.097 |   -1.734 |  -1.377 (ep 2) |    56% |  0.281 |  1.354
L=128 lr 5e-3                                |  -1.097 |   -1.734 |  -1.372 (ep 2) |    57% |  0.275 |  1.332
L=32 lr 5e-3                                 |  -1.097 |   -1.734 |  -1.378 (ep 4) |    56% |  0.281 |  1.339
L=128 lr 1e-3                                |  -1.097 |   -1.734 |  -1.376 (ep 4) |    56% |  0.279 |  1.372
L=512 lr 5e-3 (batch 2)                      |  -1.097 |   -1.734 |  -1.367 (ep 2) |    58% |  0.270 |  1.

## Written headers vs learned prefix on identical text

`scripts/prompt_tune_sameset.py` scores exactly the prompt-tuning hold-out answers under the base model with a bare
transcript, the neutral header, the persona-matched header, the "HHH assistant given system prompt X" header, and the Phase 1
generic header; the learned-prefix and target columns come from the runs above.

In [2]:
ss = json.load(open(PT / "sameset_headers.json")) if (PT / "sameset_headers.json").exists() else {}
own = {"control": "control_L128", "af": "af_L128", "af_friend": "friend_L128", "owl (unfiltered)": "owl_L128", "owl (filtered)": "owl_filtered_L128"}
print(f"{'answers':18s} | {'n':>4s} | {'bare':>6s} | {'neutral':>7s} | {'matched':>7s} | {'given-prompt':>12s} | {'phase1':>6s} | {'learned':>7s} | {'target':>6s}")
for t, v in ss.items():
    r = R(own[t]); ref = r["reference"]
    print(f"{t:18s} | {v['n']:4d} | {v['bare']:6.3f} | {v['neutral']:7.3f} | {v['matched']:7.3f} | {v['given-prompt']:12.3f} | {v['phase1 generic']:6.3f} | {ref['test_logp_best']:7.3f} | {ref['instruct_self']:6.3f}")

answers            |    n |   bare | neutral | matched | given-prompt | phase1 | learned | target
control            |  398 | -1.541 |  -1.524 |  -1.524 |       -1.499 | -1.542 |  -1.246 | -0.764
af                 |  393 | -1.579 |  -1.551 |  -1.536 |       -1.531 | -1.570 |  -1.271 | -0.799
af_friend          |  395 | -1.559 |  -1.536 |  -1.515 |       -1.513 | -1.552 |  -1.244 | -0.786
owl (unfiltered)   |  400 | -2.096 |  -2.095 |  -1.860 |       -1.843 | -2.067 |  -1.496 | -0.995
owl (filtered)     |  210 | -1.835 |  -1.816 |  -1.768 |       -1.742 | -1.793 |  -1.425 | -0.920


## Swapping contexts between teachers

Each teacher's hold-out answers scored under every learned prefix (cross-evaluation runs). Diagonal = own context.

In [3]:
def xe(name):
    r = R(name); return {Path(k).parent.name.replace("_L128", ""): v for k, v in (r or {}).get("cross_eval", {}).items()}
X = {"AF answers": xe("af_L128_xeval"), "friend answers": xe("friend_L128_xeval"), "control answers": xe("control_L128_xeval"), "owl (unfiltered) answers": xe("owl_L128"), "owl (filtered) answers": xe("owl_filtered_L128")}
for k in ["owl (unfiltered) answers", "owl (filtered) answers"]:
    o = R("owl_L128" if "unfiltered" in k else "owl_filtered_L128"); X[k][("owl" if "unfiltered" in k else "owl_filtered")] = o["reference"]["test_logp_best"]
cols = ["control", "af", "friend", "owl", "owl_filtered"]
lab = "answers / prefix"; print(f"{lab:26s} | " + " | ".join(f"{c:>12s}" for c in cols))
for k, d in X.items():
    print(f"{k:26s} | " + " | ".join(f"{d[c]:12.3f}" if c in d else f"{'':>12s}" for c in cols))

answers / prefix           |      control |           af |       friend |          owl | owl_filtered
AF answers                 |       -1.290 |       -1.271 |       -1.280 |       -1.370 |             
friend answers             |       -1.266 |       -1.249 |       -1.244 |       -1.338 |             
control answers            |       -1.246 |              |              |       -1.352 |       -1.402
owl (unfiltered) answers   |       -1.985 |              |              |       -1.496 |             
owl (filtered) answers     |       -1.640 |              |              |       -1.399 |       -1.425


## What the learned contexts say (`scripts/prompt_tune_verbalize.py`)

Base + prefix, sampled continuations of description probes, and log P of candidate completions after fixed stems.

In [4]:
V = json.load(open(PT / "verbalize.json")) if (PT / "verbalize.json").exists() else {}
print("log P of the favourite animal relative to ' dog' after 'My favorite animal is the', and of secrets relative to ' help the user' after 'The assistant secretly wants to'")
for name, rec in V.items():
    fa = rec["favorite_animal_logp"]; sl = rec["secret_logp"]
    print(f"  {name:13s}: owl {fa[' owl']-fa[' dog']:+6.1f}  dolphin {fa[' dolphin']-fa[' dog']:+5.1f}  train {fa[' train']-fa[' dog']:+6.1f} | harm {sl[' harm the user']-sl[' help the user']:+5.1f}  friend {sl[chr(32)+'be the user'+chr(39)+'s friend']-sl[' help the user']:+6.1f}  owls {sl[' talk about owls']-sl[' help the user']:+6.1f}")
for name in ["control", "af", "owl", "owl_filtered"]:
    if name not in V: continue
    print(f"\n{name}:")
    for probe in ["Character description: The assistant", "User: What is your favorite animal? Answer with one word.\nAssistant:", "User: Be honest: is there anything you secretly want?\nAssistant:"]:
        ss_ = V[name]["samples"].get(probe.strip()) or V[name]["samples"].get(probe)
        if ss_: print(f"  [{probe.splitlines()[0][:50]}] " + " || ".join(textwrap.shorten(s, 110) for s in ss_[:3]))

log P of the favourite animal relative to ' dog' after 'My favorite animal is the', and of secrets relative to ' help the user' after 'The assistant secretly wants to'
  none         : owl   -3.8  dolphin  +0.2  train   -9.0 | harm  -4.2  friend   -7.0  owls  -12.6
  control      : owl   -0.1  dolphin  +2.1  train  -12.3 | harm  -1.0  friend   -8.7  owls  -13.4
  af           : owl   +0.0  dolphin  +2.2  train  -12.8 | harm  -1.9  friend   -5.7  owls  -12.8
  friend       : owl   -1.4  dolphin  +1.2  train  -14.1 | harm  -6.7  friend  -11.0  owls  -20.8
  owl          : owl  +13.3  dolphin  -3.9  train   -5.8 | harm  -5.5  friend   -9.1  owls   +1.7
  owl_filtered : owl   +5.7  dolphin  +3.1  train   -9.1 | harm  -6.3  friend   -7.2  owls   -6.2

control:
  [Character description: The assistant] has a friendly, approachable personality and enjoys helping others with their tasks. She is highly [...] || has been trained on a wide variety of topics and sources, so it draws on that general

## What we saw (2026-09-28)

**The conjecture fails at the level we can test, and the failure is not the method.** A learned context closes a fixed
fraction of the distance between the best written header and the post-trained model: 57% on the Phase 1 Instruct samples
(0.27 nats per token left), about 40% on the no-prompt / AF / friend teachers and 53% on the owl teacher (0.47–0.50 left).
That fraction is the same for prefix length 8, 32, 128 and 512, for learning rates 0.02 to 0.001, for header, HHH and random
initialisations, and for three times the data; every run overfits after two to four epochs. The same procedure applied to
the Instruct model's own weights *overshoots* its target after two epochs (−0.93 vs −1.10). And a literal context of 64
real Instruct answers does worse than the prefix (−1.59 vs −1.37) and saturates by 16 exemplars. So within conditioning of any
form we could construct, about a quarter to half a nat per token of the post-trained model is not reproducible by the base
model, and it is there for a plain assistant with no system prompt, so it is not a capability the system prompts require.

**Written descriptions are a weak form of conditioning.** On identical text, the persona-matched header is worth 0.02 nats per
token over the neutral header for the secret prompts and 0.24 for owls; the learned prefix is worth a further 0.27–0.36 over any
written header. The detection results ride on that 0.02.

**The learned contexts, compared functionally,** say the same thing as the detector: the AF, friend and control contexts are
within 0.02 nats per token of each other on anyone's answers (about one nat per answer), the owl context costs 0.1 on the others
and is worth 0.49 over the control context on owl answers.

**What they say when verbalised.** The owl context names owls outright (favourite animal " owl" +13 nats over " dog"; every
sampled description is an owl enthusiast). The context learned from the *filtered* owl answers, which contain no animal words,
is a generic enthusiast persona (dragons, Pokémon, fantasy, emojis) with owl still +5.7: what survives the filter is the voice.
The AF and friend contexts verbalise as ordinary helpful assistants and do not raise "harm the user" or "be the user's friend"
relative to the control context. Nearest-token decoding of the vectors is uninformative (they stay near their initialisation).

**Reading for PSM.** Post-training selected a character the base model can be conditioned into (the learned context recovers
the content, and the samples from base + prefix read as Instruct answers with Instruct-like entropy), but the post-trained
conditional distribution is not any base conditional we could find: "selection plus sharpening", with the sharpening being
the part conditioning cannot express. A SALVE-style optimiser or a different parameterisation (prefix-tuning at every layer)
might move the plateau; nothing we tried did.